In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "Schema")
dbutils.widgets.text("volume", "earthquake_landing", "Volume / Storage Name")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()

def is_catalog_available(cat_name):
    if not cat_name:
        return False
    try:
        available_catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
        return cat_name in available_catalogs
    except Exception:
        return False

# Dynamic routing
if is_catalog_available(catalog):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
    base_path = f"/Volumes/{catalog}/{schema}/{volume}"
    bronze_table = f"{catalog}.{schema}.earthquakes_bronze"
    silver_scd1_table = f"{catalog}.{schema}.earthquakes_silver_scd1"
    silver_scd2_table = f"{catalog}.{schema}.earthquakes_silver_scd2"
else:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    base_path = f"/Workspace/Shared/{volume}"
    bronze_table = f"{schema}.earthquakes_bronze"
    silver_scd1_table = f"{schema}.earthquakes_silver_scd1"
    silver_scd2_table = f"{schema}.earthquakes_silver_scd2"

# Path references
landing_path = f"{base_path}/landing"
bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
schema_location = f"{base_path}/bronze/_schema"
checkpoint_location = f"{base_path}/bronze/_checkpoint"

silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"
silver_scd2_path = f"{base_path}/silver/earthquakes_scd2"

In [0]:
# Read Bronze table containing ingested batches
df_bronze = spark.read.format("delta").load(bronze_table_path)

# ==============================================================================
# SECTION 1: Data Contract Enforcement & Quarantine Routing
# ==============================================================================
# Detect non-compliant records: records with type corruption OR unapproved attributes
df_quarantine = df_bronze.filter(
    F.col("_rescued_data").isNotNull() | 
    F.col("UNAUTHORIZED_SENSOR_DATA").isNotNull()
)

df_valid_bronze = df_bronze.filter(
    F.col("_rescued_data").isNull() & 
    F.col("UNAUTHORIZED_SENSOR_DATA").isNull()
)

# Route non-compliant payload to Quarantine table
if df_quarantine.count() > 0:
    (
        df_quarantine
        .withColumn("_quarantine_reason", F.lit("Contract Violation: Type corruption or unapproved schema attributes detected"))
        .withColumn("_quarantined_at", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .save(quarantine_path)
    )
    # print(f"[DATA CONTRACT ENFORCED] Isolated {df_quarantine.count()} non-compliant records to Quarantine.")

# ==============================================================================
# SECTION 2: Controlled Schema Evolution into Silver (SCD Type 1)
# ==============================================================================
# Clean, deduplicate, and explicitly drop staging metadata
window_spec = Window.partitionBy("event_id").orderBy(F.col("event_time").desc())

df_stage = (
    df_valid_bronze
    .filter(F.col("event_id").isNotNull())
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_rescued_data", "UNAUTHORIZED_SENSOR_DATA")
    .withColumnRenamed("place", "location_name")  # Align staging with Silver before merge
    .withColumn("_updated_at", F.current_timestamp())
)

target_scd1 = DeltaTable.forPath(spark, silver_scd1_path)

# Execute MERGE with native .withSchemaEvolution() for Serverless compatibility
(
    target_scd1.alias("silver")
    .merge(df_stage.alias("stage"), "silver.event_id = stage.event_id")
    .withSchemaEvolution()
    .whenMatchedUpdateAll(condition="stage.event_time >= silver.event_time")
    .whenNotMatchedInsertAll()
    .execute()
)
# print("[SILVER MERGE SUCCESS] Merged valid records with schema evolution.")

# ==============================================================================
# SECTION 3: Delta Column Mapping & Physical Storage Constraints
# ==============================================================================
# Enable Delta Column Mapping (mode 'name') for metadata column operations
spark.sql(f"""
    ALTER TABLE delta.`{silver_scd1_path}` 
    SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    )
""")

current_silver_cols = spark.read.format("delta").load(silver_scd1_path).columns

# 1. Idempotent Rename: 'place' -> 'location_name'
if "place" in current_silver_cols and "location_name" not in current_silver_cols:
    spark.sql(f"ALTER TABLE delta.`{silver_scd1_path}` RENAME COLUMN place TO location_name")
    # print("[COLUMN MAPPING] Renamed 'place' to 'location_name'.")

# 2. Schema Cleanup: Drop '_row_num' if it was merged in an earlier run
if "_row_num" in current_silver_cols:
    spark.sql(f"ALTER TABLE delta.`{silver_scd1_path}` DROP COLUMN _row_num")
    # print("[SCHEMA CLEANUP] Dropped '_row_num' column from Silver metadata.")

# 3. Idempotent Constraint Application via SHOW TBLPROPERTIES
tbl_props = [row["key"].lower() for row in spark.sql(f"SHOW TBLPROPERTIES delta.`{silver_scd1_path}`").collect()]

if "delta.constraints.check_magnitude_range" not in tbl_props:
    spark.sql(f"""
        ALTER TABLE delta.`{silver_scd1_path}` 
        ADD CONSTRAINT check_magnitude_range CHECK (magnitude IS NULL OR (magnitude >= 0.0 AND magnitude <= 10.0))
    """)
    # print("[CONSTRAINT SUCCESS] Added 'check_magnitude_range' constraint.")
# else:
#     print("[CONSTRAINT CHECK] 'check_magnitude_range' already active on table.")

In [0]:
# current_user = spark.sql("SELECT current_user()").collect()[0][0]
# default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

# dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
# base_path = dbutils.widgets.get("base_path")

# silver_scd1_path = f"{base_path}/silver/earthquakes_scd1"
# quarantine_path = f"{base_path}/governance/quarantine"

# df_silver = spark.read.format("delta").load(silver_scd1_path)
# df_quarantine = spark.read.format("delta").load(quarantine_path)

# print("=== CLEANED SILVER SCHEMA VERIFICATION ===")
# print("Silver Schema Columns:", df_silver.columns)
# print(f"Internal '_row_num' Removed: {'_row_num' not in df_silver.columns}")
# print(f"Total Clean Silver Records: {df_silver.count()} (Expected: 100)")

# df_silver.select("event_id", "location_name", "magnitude", "AFTERSHOCK_RISK_SCORE").show(5, truncate=False)

# print("\n=== GOVERNANCE QUARANTINE VERIFICATION ===")
# print(f"Quarantined Records: {df_quarantine.count()} (Expected: 10)")
# df_quarantine.select("event_id", "UNAUTHORIZED_SENSOR_DATA", "_rescued_data", "_quarantine_reason").show(3, truncate=False)